In [2]:
"""
draw_mfg_regretnet.py
MFG-RegretNet Network Architecture Diagram
Paper: "Privacy as Commodity: MFG-RegretNet for Large-Scale
        Privacy Trading in Federated Learning"

Usage:
    pip install matplotlib numpy
    python draw_mfg_regretnet.py
Outputs:
    mfg_regretnet_arch.pdf  (300 dpi, vector)
    mfg_regretnet_arch.png  (200 dpi, raster)
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Circle
import numpy as np

# ══════════════════════════════════════════════════════════════════════
# 1. Canvas
# ══════════════════════════════════════════════════════════════════════
FW, FH = 26, 15
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, FW)
ax.set_ylim(0, FH)
ax.axis('off')
fig.patch.set_facecolor('white')

# ══════════════════════════════════════════════════════════════════════
# 2. Colour Palette
# ══════════════════════════════════════════════════════════════════════
C = dict(
    bid='#D6EAF8',   # light blue   – bid inputs
    mfg='#FEF9E7',   # cream        – MFG block
    ahn='#FDEBD0',   # peach        – Auction Header Network
    alc='#FADBD8',   # blush pink   – Allocation Network
    pay='#D6EAF8',   # sky blue     – Payment Network
    rgt='#E8DAEF',   # lavender     – Regret box
    out='#D5F5E3',   # mint green   – output blocks
    al ='#F4ECF7',   # pale purple  – Augmented Lagrangian
    wht='#FFFFFF',   # white        – generic nodes
    gld='#784212',   # amber        – b_MFG highlights
    red='#C0392B',   # red          – allocation / softmax
    blu='#1A5276',   # deep blue    – payment / sigmoid
    grn='#1E8449',   # dark green   – output / utility
    pur='#6C3483',   # purple       – regret / AL
    nvy='#2C3E50',   # dark navy    – default arrows
    gry='#888888',   # grey         – borders
    org='#E67E22',   # orange       – AHN border
)

# ══════════════════════════════════════════════════════════════════════
# 3. Drawing Primitives
# ══════════════════════════════════════════════════════════════════════

def rbox(cx, cy, w, h, fc, ec=None, lw=1.5, z=2, rad=0.12):
    """Rounded-corner rectangle centred at (cx, cy)."""
    ax.add_patch(FancyBboxPatch(
        (cx - w / 2, cy - h / 2), w, h,
        boxstyle=f'round,pad={rad}',
        facecolor=fc, edgecolor=ec or C['gry'],
        linewidth=lw, zorder=z))


def txt(cx, cy, s, fs=9, fw='normal', col='#111111',
        ha='center', va='center', z=5, it=False):
    """Place a text label."""
    ax.text(cx, cy, s,
            fontsize=fs, fontweight=fw, color=col,
            ha=ha, va=va, zorder=z,
            style='italic' if it else 'normal')


def node(cx, cy, r=0.27, lbl='', fc=None, ec=None, fs=8, z=4):
    """Filled circle node with optional centre label."""
    ax.add_patch(Circle((cx, cy), r,
                         facecolor=fc or C['wht'],
                         edgecolor=ec or C['gry'],
                         linewidth=1.2, zorder=z))
    if lbl:
        ax.text(cx, cy, lbl, ha='center', va='center',
                fontsize=fs, zorder=z + 1, color='#111111')


def arr(x1, y1, x2, y2, col=None, lw=1.5, rad=0.0, sty='->'):
    """Annotated arrow from (x1,y1) to (x2,y2)."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(
                    arrowstyle=sty, color=col or C['nvy'], lw=lw,
                    connectionstyle=f'arc3,rad={rad}'),
                zorder=7)


def thin_line(x1, y1, x2, y2, col='#CCCCCC', lw=0.65, ls='-'):
    """Thin connector line (no arrow head)."""
    ax.plot([x1, x2], [y1, y2], ls, color=col, lw=lw, zorder=3)


def fc_conn(x_l, y_list_l, x_r, y_list_r, col='#CCCCCC', lw=0.65):
    """All-to-all thin connections between two layers."""
    for yl in y_list_l:
        for yr in y_list_r:
            thin_line(x_l, yl, x_r, yr, col=col, lw=lw)


# ══════════════════════════════════════════════════════════════════════
# 4. Layout Constants
# ══════════════════════════════════════════════════════════════════════
# ── Block centres ──────────────────────────────────────────────────
BX,  BY  =  1.3,  8.5    # Bid Inputs
MX,  MY  =  3.9,  8.5    # MFG System
AHN_CX   =  7.3           # AHN centre-x (cy same as others = 8.5)
AHN_CY   =  8.5
ACX, ACY = 13.1, 11.3    # Allocation Network
RCX, RCY = 13.1,  8.0    # Regret Box
PCX, PCY = 13.1,  4.5    # Payment Network
GX,  GY  = 18.0, 11.3   # g_i computation
PiX, PiY = 18.0,  4.5   # p_i computation
UX,  UY  = 22.0, 11.3   # Final utility u_i
PbX, PbY = 22.0,  4.5   # BF-projected payment \bar p_i
ALX, ALY = 22.0,  8.0   # Augmented Lagrangian Loss

# ── AHN internal columns ───────────────────────────────────────────
Aa_x  = 5.6    # input a-nodes
Ah1_x = 6.9    # hidden h
Ah2_x = 8.0    # hidden h'
Ac_x  = 9.4    # context c-nodes (AHN output)

# ── Allocation Net internal columns ───────────────────────────────
ANh_x = 11.4   # hidden h^(a)
ANs_x = 13.1   # softmax column
ANz_x = 14.7   # z output nodes

# ── Payment Net internal columns ──────────────────────────────────
PNh_x = 11.4   # hidden h^(p)
PNs_x = 13.1   # sigmoid column
PNp_x = 14.7   # p output nodes

# ── Node y-positions ──────────────────────────────────────────────
# Bids (with MFG badge at bottom of block)
bid_ys  = [12.5, 11.7, 10.9, 10.1, 9.3, 8.5, 7.7, 6.9, 6.1, 5.3]
bid_lbs = [r'$b_1$', r'$\vdots$', r'$b_i$', r'$\vdots$', r'$b_n$',
           '', r'$b_{\mathrm{MFG}}$', r'$\uparrow$ mean-field stat', '', '']

# AHN node rows
a_ys  = [12.0, 11.1, 10.2, 9.3, 8.4]
h_ys  = [12.0, 11.1, 10.2, 9.3, 8.4]   # h and h' share same y-rows
# Context:  half above regret, half below (avoids visual overlap with RGT box)
c_ys  = [11.8, 10.5, 6.2, 5.0]

# Allocation Network
ah_ys = [12.0, 11.3, 10.6]
sm_ys = [12.0, 11.3, 10.6]
z_ys  = [12.4, 11.9, 11.4, 10.9, 10.4]

# Payment Network
ph_ys = [5.2, 4.5, 3.8]
sg_ys = [5.2, 4.5, 3.8]
pp_ys = [5.6, 5.1, 4.6, 4.1, 3.6]

# Active (non-vdots) members for FC connections
act_a    = [y for lbl, y in zip(
    [r'$a_1$', r'$\vdots$', r'$a_i$', r'$a_k$', r"$a'$"], a_ys)
    if 'vdots' not in lbl]
act_h    = h_ys[::2]            # representative subset
act_c_up = c_ys[:2]             # upper context → allocation
act_c_dn = c_ys[2:]             # lower context → payment
act_z    = z_ys[::2]
act_p    = pp_ys[::2]

# ══════════════════════════════════════════════════════════════════════
# 5. BLOCK 1 · Bid Inputs
# ══════════════════════════════════════════════════════════════════════
rbox(BX, BY, 2.3, 9.5, C['bid'], ec='#2980B9', lw=1.8, z=1)
txt(BX, 13.55, 'Bid Input', fs=10, fw='bold')
txt(BX, 13.1, r'$\mathbf{b}\in\mathbb{R}^{N\times m}$', fs=8.5, it=True)

a_lbls_full = [r'$a_1$', r'$\vdots$', r'$a_i$', r'$a_k$', r"$a'$"]

for y, lbl in zip(bid_ys, bid_lbs):
    if not lbl:
        continue
    elif 'vdots' in lbl:
        txt(BX, y, lbl, fs=13)
    elif 'uparrow' in lbl or 'mean' in lbl:
        txt(BX, y, lbl, fs=7.5, col=C['gld'], it=True)
    elif 'MFG' in lbl:
        rbox(BX, y, 1.7, 0.58, '#FEF3CD', ec='#F39C12', lw=2, z=3, rad=0.08)
        txt(BX, y, lbl, fs=9.5, fw='bold', col=C['gld'])
    else:
        rbox(BX, y, 1.3, 0.55, C['wht'], ec='#7FB3D3', lw=1.3, z=3, rad=0.08)
        txt(BX, y, lbl, fs=10.5, fw='bold')

# ══════════════════════════════════════════════════════════════════════
# 6. BLOCK 2 · MFG System
# ══════════════════════════════════════════════════════════════════════
rbox(MX, MY, 2.5, 9.5, C['mfg'], ec='#F39C12', lw=1.8, z=1)
txt(MX, 13.55, 'MFG System', fs=10, fw='bold')

# MFE solver badge
rbox(MX, 12.0, 2.1, 1.3, '#FFFDE7', ec='#F39C12', lw=2, z=3)
txt(MX, 12.35, r'MFE: $\mu_t^*\in\mathcal{P}(\mathcal{S})$', fs=8)
txt(MX, 11.85, r'HJB $\leftrightarrow$ FP solve', fs=7.8, it=True, col='#7D6608')

# b_MFG formula box
rbox(MX, 10.35, 2.1, 0.7, '#FEF3CD', ec='#F39C12', lw=2, z=3)
txt(MX, 10.35,
    r'$b_{\mathrm{MFG}}=\tfrac{1}{N}\textstyle\sum_{i}b_i$',
    fs=8, col=C['gld'])

# b_MFG output labelled box
rbox(MX, 9.1, 1.9, 0.62, '#FDF3E3', ec=C['gld'], lw=2.2, z=3)
txt(MX, 9.1, r'$b_{\mathrm{MFG}}$', fs=10, fw='bold', col=C['gld'])

# ε_N approximation guarantee
rbox(MX, 7.8, 2.1, 0.85, '#EAFAF1', ec='#27AE60', lw=1.5, z=3)
txt(MX, 8.05, r'Approx error:', fs=7.5)
txt(MX, 7.62, r'$\varepsilon_N=\mathcal{O}(N^{-1/2})$', fs=8, col=C['grn'])

# Arrows: bids → MFG, internal MFG flow
for by in [12.5, 10.9, 9.3, 7.7]:
    arr(BX + 1.15, by, MX - 1.25, by)
arr(MX, 11.3, MX, 11.0, col='#F39C12', lw=1.5)  # MFE → formula
arr(MX, 9.7,  MX, 9.4, col=C['gld'], lw=1.5)     # formula → output

# ══════════════════════════════════════════════════════════════════════
# 7. BLOCK 3 · Auction Header Network (AHN)
# ══════════════════════════════════════════════════════════════════════
rbox(AHN_CX, AHN_CY, 5.0, 9.5, C['ahn'], ec=C['org'], lw=1.8, z=1)
txt(AHN_CX, 13.55, 'Auction Header Network  (AHN)', fs=10, fw='bold')
txt(AHN_CX, 13.1,
    r'Params $\theta_h$ | Input: $[\,b_1,\ldots,b_N,\;b_{\mathrm{MFG}}\,]$',
    fs=8.5, it=True)

# ── a-nodes (AHN input) ──────────────────────────────────────────
a_node_lbs = [r'$a_1$', r'$\vdots$', r'$a_i$', r'$a_k$', r"$a'$"]
for lbl, y in zip(a_node_lbs, a_ys):
    if 'vdots' in lbl:
        txt(Aa_x, y, lbl, fs=13)
    else:
        fc_n = '#FEF3CD' if "'" in lbl else C['wht']
        ec_n = C['gld']  if "'" in lbl else C['wht']

Error in callback <function _draw_all_if_interactive at 0x7c4b3e406ca0> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
b_{\mathrm{MFG}}=\tfrac{1}{N}\textstyle\sum_{i}b_i
                 ^
ParseFatalException: Unknown symbol: \tfrac, found '\'  (at char 17), (line:1, col:18)

ValueError: 
b_{\mathrm{MFG}}=\tfrac{1}{N}\textstyle\sum_{i}b_i
                 ^
ParseFatalException: Unknown symbol: \tfrac, found '\'  (at char 17), (line:1, col:18)

<Figure size 2600x1500 with 1 Axes>